# 🤖 Intelligent Agents Lab: Vacuum World

An agent perceives its environment through **sensors** and acts on it through **actuators**. In this notebook we build a Vacuum World, compare two types of agents, then describe agents with PEAS and classify environments.


This is a classic AI problem. The environment has rooms (A, B), each either Dirty or Clean. The agent (the vacuum) stands in one room and can suck, or move left/right. The goal is to clean all rooms.


# **Import Libraries**

In [ ]:
import random

We import Python's random library because the class needs random numbers to decide whether a room gets dirty again or whether sucking fails






* Randomness turns an ideal environment into a realistic one.
___________________________________

#1_ Defining the VacuumEnv class and its __init__ constructor







In [ ]:


class VacuumEnv:
    def __init__(
        self,
        rooms=("A", "B"),
        dirty=("A", "B"),
        start="A",
        p_dirty_again=0.0,
        p_suck_fail=0.0,
        seed=None
    ):
        self.rooms = list(rooms)

        self.status = {
            room: ("Dirty" if room in dirty else "Clean")
            for room in self.rooms
        }

        self.loc = start
        self.p_dirty_again = p_dirty_again
        self.p_suck_fail = p_suck_fail
        self.rng = random.Random(seed)

    # Agent receives a percept
    def percept(self):
        return self.loc, self.status[self.loc]

    # Environment executes the agent's action
    def step(self, action):
        i = self.rooms.index(self.loc)
        move_cost = 0.0

        if action == "Suck":
            if self.rng.random() >= self.p_suck_fail:
                self.status[self.loc] = "Clean"

        elif action == "Right" and i < len(self.rooms) - 1:
            self.loc = self.rooms[i + 1]
            move_cost = 0.5

        elif action == "Left" and i > 0:
            self.loc = self.rooms[i - 1]
            move_cost = 0.5

        # Clean rooms may become dirty again
        for room in self.rooms:
            if (
                self.status[room] == "Clean"
                and self.rng.random() < self.p_dirty_again
            ):
                self.status[room] = "Dirty"

        clean_rooms = sum(
            status == "Clean"
            for status in self.status.values()
        )

        return clean_rooms - move_cost

> 💡 **Insight**

>The class keeps the entire world state in one place.

> The agent sees only a small part of the world, which is the root of the difficulty.

> The reward encourages cleanliness and penalizes unnecessary movement.

# Running the agent





In [ ]:
def run(agent, env, steps=20, verbose=False):
    total = 0

    for t in range(steps):
        percept = env.percept()
        action = agent(percept)
        reward = env.step(action)

        total += reward

        if verbose:
            print(
                f"t={t:2d} "
                f"percept={str(percept):18} "
                f"action={action:6} "
                f"reward={reward:4} "
                f"status={env.status}"
            )

    return total



# Simple reflex agent

In [ ]:
def reflex_vacuum(percept):
    location, status = percept

    if status == "Dirty":
        return "Suck"

    elif location == "A":
        return "Right"

    else:
        return "Left"

#Model-based agent

In [ ]:
class ModelBasedVacuum:
    def __init__(self, rooms=("A", "B")):
        self.rooms = list(rooms)
        self.known_clean = set()

    def __call__(self, percept):
        location, status = percept

        if status == "Dirty":
            self.known_clean.discard(location)
            return "Suck"

        self.known_clean.add(location)

        if len(self.known_clean) == len(self.rooms):
            return "NoOp"

        return "Right" if location == "A" else "Left"

#Testing both agents

In [ ]:
print("=== Reflex ===")
print("score:", run(reflex_vacuum, VacuumEnv(), steps=10, verbose=True))
print("\n=== Model-based ===")
print("score:", run(ModelBasedVacuum(), VacuumEnv(), steps=10, verbose=True))

=== Reflex ===
t= 0 percept=('A', 'Dirty')     action=Suck   reward= 1.0 status={'A': 'Clean', 'B': 'Dirty'}
t= 1 percept=('A', 'Clean')     action=Right  reward= 0.5 status={'A': 'Clean', 'B': 'Dirty'}
t= 2 percept=('B', 'Dirty')     action=Suck   reward= 2.0 status={'A': 'Clean', 'B': 'Clean'}
t= 3 percept=('B', 'Clean')     action=Left   reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 4 percept=('A', 'Clean')     action=Right  reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 5 percept=('B', 'Clean')     action=Left   reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 6 percept=('A', 'Clean')     action=Right  reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 7 percept=('B', 'Clean')     action=Left   reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 8 percept=('A', 'Clean')     action=Right  reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
t= 9 percept=('B', 'Clean')     action=Left   reward= 1.5 status={'A': 'Clean', 'B': 'Clean'}
score: 14.0

=== Model-based ===
t= 0 percept

#Performance evaluation and comparison

In [ ]:
def evaluate(agent_factory, steps=50, runs=200, **env_kw):
    return sum(run(agent_factory(), VacuumEnv(seed=s, **env_kw), steps) for s in range(runs)) / runs

settings = {
    "Deterministic":                dict(),
    "Dirt comes back (p=0.10)":     dict(p_dirty_again=0.10),
    "Suck fails (p=0.30)":          dict(p_suck_fail=0.30),
}
print(f"{'environment':28}{'Reflex':>10}{'Model-based':>14}")
for name, kw in settings.items():
    r = evaluate(lambda: reflex_vacuum, **kw)
    m = evaluate(lambda: ModelBasedVacuum(), **kw)
    print(f"{name:28}{r:10.2f}{m:14.2f}")

environment                     Reflex   Model-based
Deterministic                    74.00         97.50
Dirt comes back (p=0.10)         63.41         53.04
Suck fails (p=0.30)              73.23         96.34


> 💡 **Insight**

> - An average over many runs is more reliable than a single result.
>
> -  No agent wins in every environment; the choice depends on the environment.

PEAS descriptions

In [ ]:
peas = {
    "Language tutor": {
        "Performance": "Learning progress and correct answers",
        "Environment": "Student and learning interface",
        "Actuators": "Display questions, feedback, explanations",
        "Sensors": "Student answers and interaction"
    },

    "Delivery drone": {
        "Performance": "Safe and accurate delivery",
        "Environment": "Buildings, roads, weather, delivery locations",
        "Actuators": "Motors, propellers, package release",
        "Sensors": "GPS, camera, distance and obstacle sensors"
    }
}

for agent, details in peas.items():
    print(f"\n{agent}")
    for component, description in details.items():
        print(f"{component}: {description}")


Language tutor
Performance: Learning progress and correct answers
Environment: Student and learning interface
Actuators: Display questions, feedback, explanations
Sensors: Student answers and interaction

Delivery drone
Performance: Safe and accurate delivery
Environment: Buildings, roads, weather, delivery locations
Actuators: Motors, propellers, package release
Sensors: GPS, camera, distance and obstacle sensors


Environment classification

In [ ]:
tasks = {
    "Sudoku": {
        "Observable": "Fully",
        "Agents": "Single",
        "Determinism": "Deterministic",
        "Structure": "Sequential",
        "Dynamics": "Static",
        "Type": "Discrete"
    },

    "Rock-Paper-Scissors": {
        "Observable": "Fully, after both moves",
        "Agents": "Multi",
        "Determinism": "Stochastic from one player's perspective",
        "Structure": "Episodic",
        "Dynamics": "Static during each round",
        "Type": "Discrete"
    },

    "Image classifier": {
        "Observable": "Fully, for the provided image",
        "Agents": "Single",
        "Determinism": "Deterministic, for a fixed model",
        "Structure": "Episodic",
        "Dynamics": "Static",
        "Type": "Continuous-valued image input"
    }
}

for task, properties in tasks.items():
    print(f"\n{task}")
    for property_name, value in properties.items():
        print(f"{property_name}: {value}")


Sudoku
Observable: Fully
Agents: Single
Determinism: Deterministic
Structure: Sequential
Dynamics: Static
Type: Discrete

Rock-Paper-Scissors
Observable: Fully, after both moves
Agents: Multi
Determinism: Stochastic from one player's perspective
Structure: Episodic
Dynamics: Static during each round
Type: Discrete

Image classifier
Observable: Fully, for the provided image
Agents: Single
Determinism: Deterministic, for a fixed model
Structure: Episodic
Dynamics: Static
Type: Continuous-valued image input


> 💡 **Insight**
> - Classifying the environment decides which kind of agent suits it.